#  Capstone Project 2  
## Semiconductor Manufacturing Process – Pass/Fail Prediction


## 1. Project Objective

The objective of this project is to build a supervised machine learning model to predict the Pass/Fail yield of a semiconductor manufacturing process using sensor data.


## 2. Data Import and Library Setup

This section imports all the required libraries for data manipulation, visualization, preprocessing, and machine learning model development.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

import joblib


: 

## 2. Data Import

The dataset is loaded into the notebook using pandas. The shape of the dataset is examined to understand the number of observations and features.


In [ ]:
df = pd.read_csv("/content/signal-data.csv")
df.shape


## 2. Data Understanding

This section explores the structure of the dataset, including feature types, target variable, and basic statistics.


In [ ]:
df.head()

**Observation:**  
The dataset consists of time-stamped sensor measurements along with a binary Pass/Fail target variable. The large number of sensor features highlights the high-dimensional nature of the data.


The sensor columns are renamed to improve readability and make further analysis easier.


In [ ]:
total_columns = df.shape[1]

sensor_columns = (["Time"] +
                  [f"Sensor_{i}" for i in range(1, total_columns-1)] +
                  ["Pass/Fail"])

df.columns = sensor_columns


This cell displays the dataset after renaming columns, allowing a visual inspection of sensor values and the target variable.


In [ ]:
df

The Time column is converted to a datetime format to ensure correct data type handling. Data types of all features are checked to confirm consistency.


In [ ]:
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')


In [ ]:
df.dtypes

This step identifies missing values across all sensor features to assess the extent of data incompleteness.


In [ ]:
df.isna().sum()


### Missing Value Treatment

Missing values in numerical sensor features are handled using median imputation to reduce the influence of extreme values.


In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = numeric_cols.drop('Pass/Fail')
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())





This step verifies that all missing values in numerical sensor features have been successfully handled.


In [ ]:
df[numeric_cols].isna().sum()


## Target Variable Distribution

The distribution of the Pass/Fail target variable is examined to identify potential class imbalance.


In [ ]:
df['Pass/Fail'].value_counts()


**Observation:**  
The dataset is highly imbalanced, with significantly fewer failure cases compared to pass cases. This imbalance must be addressed during model training.


Descriptive statistics are computed to summarize the central tendency and spread of sensor measurements.


In [ ]:
df.describe().round(2)

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

## Outlier Detection and Treatment

Outliers are detected using the Interquartile Range (IQR) method and capped to reduce the influence of extreme values.


In [ ]:
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_capped = df.copy()

for col in numeric_cols:
    df_capped[col] = np.where(
        df[col] < lower_bound[col], lower_bound[col],
        np.where(df[col] > upper_bound[col], upper_bound[col], df[col])
    )


The following boxplot visualizes the distribution of selected sensor features before outlier treatment.


In [ ]:
sample_cols = numeric_cols[:10]
plt.figure(figsize=(10, 5))
df[sample_cols].boxplot()
plt.title("Before Outlier Capping")
plt.xticks(rotation=45)
plt.show()

**Observation:**  
Several features exhibit extreme values outside the typical range, confirming the presence of outliers in the dataset.


The following boxplot shows the distribution of selected sensor features after applying outlier capping.


In [ ]:
plt.figure(figsize=(10, 5))
df_capped[sample_cols].boxplot()
plt.title("After Outlier Capping")
plt.xticks(rotation=45)
plt.show()

**Observation:**  
After outlier capping, extreme values are reduced while preserving the overall distribution of the data.


A side-by-side comparison is performed to visually assess the impact of outlier capping on sensor feature distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df[sample_cols].boxplot(ax=axes[0])
axes[0].set_title("Before Capping")
axes[0].tick_params(axis='x', rotation=45)

df_capped[sample_cols].boxplot(ax=axes[1])
axes[1].set_title("After Capping")
axes[1].tick_params(axis='x', rotation=45)

plt.show()


## Feature and Target Separation

Predictor variables and the target variable are separated to prepare the dataset for model training.


In [ ]:
X = df_capped.drop(['Time', 'Pass/Fail'], axis=1)
y = df['Pass/Fail']


The distribution of the target variable is rechecked before applying class imbalance handling techniques.


In [ ]:
y.value_counts()

## Handling Class Imbalance Using SMOTE

SMOTE is applied to balance the target classes by synthetically generating minority class samples.


In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

y_resampled.value_counts()


**Observation:**  
After applying SMOTE, the dataset becomes balanced, enabling fair model training without bias toward the majority class.


## Train-Test Split

The dataset is split into training and testing sets to evaluate model performance on unseen data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.25,
    random_state=42,
    stratify=y_resampled
)


## Feature Scaling

Standardization is applied to ensure all features contribute equally during model training.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Model Training: Logistic Regression

A Logistic Regression model is trained as a baseline classifier and evaluated using accuracy and classification metrics.


In [ ]:
lr = LogisticRegression(max_iter=500)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


**Observation:**  
The Logistic Regression model achieves strong overall performance, indicating that the preprocessed features are informative for classification.


## Model Training and Hyperparameter Tuning: Random Forest

A Random Forest classifier is trained using GridSearchCV to identify optimal hyperparameters. Cross-validation is applied to improve generalization performance.


In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

cv = StratifiedKFold(n_splits=5)

grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


**Observation:**  
The Random Forest model achieves very high accuracy and balanced precision-recall scores, indicating strong performance on both classes.


## Model Training and Hyperparameter Tuning: Support Vector Machine (SVM)

An SVM classifier is trained with hyperparameter tuning using GridSearchCV to capture complex decision boundaries in the data.


In [ ]:
svc = SVC()

param_grid_svc = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

grid_svc = GridSearchCV(
    svc,
    param_grid_svc,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_svc.fit(X_train_scaled, y_train)

best_svc = grid_svc.best_estimator_

y_pred_svc = best_svc.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))


**Observation:**  
The SVM model achieves the highest accuracy among all tested models, demonstrating excellent classification performance.


## Model Comparison

The performance of all trained models is compared using accuracy to identify the best-performing classifier.


In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svc)
    ]
})

results


## Model Saving

The final selected model and the scaler are saved for future use and deployment.

In [ ]:
joblib.dump(best_svc, "final_yield_prediction_model.pkl")
joblib.dump(scaler, "scaler.pkl")


## Model Predictions

The final model is used to generate predictions on the test dataset, and results are compared with actual labels.


In [ ]:
# Predict using best model (example: SVM)
y_pred = best_svc.predict(X_test_scaled)

# Convert numeric output to readable labels
y_pred_label = pd.Series(y_pred).map({-1: "Pass", 1: "Fail"})

# Create output dataframe
output_df = pd.DataFrame({
    "Actual": pd.Series(y_test).map({-1: "Pass", 1: "Fail"}).values,
    "Predicted": y_pred_label.values
})

output_df.head()


**Observation:**  
The predicted labels closely match the actual labels, confirming the reliability of the final model.


## Example Prediction on New Data

An example prediction is performed on a new sensor input to demonstrate how the trained model can be used in real-world scenarios.


In [ ]:
# Example input (must have SAME number of features)
new_sample = X.iloc[[0]]   # replace with new sensor values

# Scale using trained scaler
new_sample_scaled = scaler.transform(new_sample)

# Predict
prediction = best_svc.predict(new_sample_scaled)[0]

# Convert to label
result = "Pass" if prediction == -1 else "Fail"

print("Prediction Result:", result)


## Final Predictions on Complete Dataset

The trained model is applied to the entire dataset to generate Pass/Fail predictions for each production instance.


In [ ]:
df_results = df.copy()

df_results["Prediction"] = best_svc.predict(scaler.transform(X))
df_results["Prediction"] = df_results["Prediction"].map({-1: "Pass", 1: "Fail"})

df_results[["Time", "Prediction"]].head(10)


**Observation:**  
The model successfully generates yield predictions for each timestamp, demonstrating its applicability for real-time monitoring and decision support in semiconductor manufacturing.

## Conclusion

This project successfully demonstrates the application of machine learning techniques to predict Pass/Fail outcomes in a semiconductor manufacturing process.  

Key takeaways include:
- Proper data preprocessing and outlier handling significantly improve model performance.
- Class imbalance handling using SMOTE is essential for fair learning.
- Among all models tested, SVM achieved the best overall performance.

Future work may include dimensionality reduction, feature selection, and real-time deployment of the model.
